In [15]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

import torch
print(torch.cuda.device_count())   # should be 1 now
print(torch.cuda.get_device_name(0))  # GPU 8 will appear as cuda:0


import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of visible GPUs: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")


1
NVIDIA L40S
CUDA available: True
Number of visible GPUs: 1
Device 0: NVIDIA L40S


In [16]:
import cv2
import numpy as np
import random
import torch
import torch.nn as nn
import gc
import os
import pandas as pd
from tqdm import tqdm
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from PIL import Image
import json
import time
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc as sklearn_auc

In [17]:
class ImagePreprocessor:
    
    def __init__(self, square_size=3052, target_size=(448, 448)):
        self.square_size = square_size
        self.target_size = target_size
    
    def analyze_image_color_distribution(self, image):
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image
        
        hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
        hist = hist.flatten()
        
        hist_for_peaks = hist.copy()
        hist_for_peaks[:21] = 0
        hist_for_peaks[235:] = 0
        
        if np.sum(hist_for_peaks) > 0:
            mode_value = np.argmax(hist_for_peaks)
        else:
            hist_fallback = hist.copy()
            hist_fallback[0] = 0
            hist_fallback[255] = 0
            
            if np.sum(hist_fallback) > 0:
                mode_value = np.argmax(hist_fallback)
            else:
                mode_value = 128
        
        if mode_value <= 20 or mode_value >= 235:
            hist_copy = hist.copy()
            hist_copy[mode_value] = 0
            hist_copy[:21] = 0
            hist_copy[235:] = 0
            
            if np.sum(hist_copy) > 0:
                next_mode = np.argmax(hist_copy)
                mode_value = next_mode
            else:
                mode_value = 128
        
        range_offset = max(int(mode_value * 0.2), 15)
        min_noise_color = max(0, mode_value - range_offset)
        max_noise_color = min(255, mode_value + range_offset)
        
        if max_noise_color - min_noise_color < 30:
            center = (min_noise_color + max_noise_color) // 2
            min_noise_color = max(0, center - 15)
            max_noise_color = min(255, center + 15)
        
        return mode_value, (min_noise_color, max_noise_color), hist
    
    def pad_to_square_and_resize(self, image_path, analyze_colors=True):
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Cannot load image: {image_path}")
        
        color_analysis = None
        if analyze_colors:
            mode_value, noise_range, hist = self.analyze_image_color_distribution(image)
            color_analysis = {
                'mode_value': mode_value,
                'noise_range': noise_range,
                'histogram': hist
            }
        
        h, w = image.shape[:2]
        max_dim = max(h, w, self.square_size)
        
        pad_h = max_dim - h
        pad_w = max_dim - w
        
        top = pad_h // 2
        bottom = pad_h - top
        left = pad_w // 2
        right = pad_w - left
        
        if len(image.shape) == 3:
            padded = cv2.copyMakeBorder(image, top, bottom, left, right, 
                                      cv2.BORDER_CONSTANT, value=[0, 0, 0])
        else:
            padded = cv2.copyMakeBorder(image, top, bottom, left, right, 
                                      cv2.BORDER_CONSTANT, value=0)
        
        resized = cv2.resize(padded, self.target_size, interpolation=cv2.INTER_LANCZOS4)
        
        return resized, color_analysis

In [18]:
class GradientNoiseInjector:
    
    def __init__(self):
        pass
    
    def generate_gradient_noise_pattern(self, color_analysis, dot_size, gradient_type='radial'):
        """
        Generate gradient noise pattern ranging from histogram-detected values (clipped to 20-235)
        gradient_type: 'radial', 'horizontal', 'vertical', 'diagonal'
        """
        if color_analysis is None:
            # Fallback to default range 20-235
            min_color, max_color = 20, 235
        else:
            min_color, max_color = color_analysis['noise_range']
            # Clip to ensure we're within 20-235
            min_color = max(20, min_color)
            max_color = min(235, max_color)
        
        # Ensure reasonable range
        if max_color - min_color < 30:
            center = (min_color + max_color) // 2
            min_color = max(20, center - 15)
            max_color = min(235, center + 15)
        
        # Create gradient pattern
        gradient = np.zeros((dot_size, dot_size), dtype=np.uint8)
        
        if gradient_type == 'radial':
            # Radial gradient from center to edges
            center_y, center_x = dot_size // 2, dot_size // 2
            max_distance = np.sqrt(center_x**2 + center_y**2)
            
            for i in range(dot_size):
                for j in range(dot_size):
                    distance = np.sqrt((i - center_y)**2 + (j - center_x)**2)
                    # Normalize distance to [0, 1]
                    norm_distance = min(distance / max_distance, 1.0)
                    # Map to color range
                    color_val = int(min_color + (max_color - min_color) * norm_distance)
                    gradient[i, j] = np.clip(color_val, min_color, max_color)
        
        elif gradient_type == 'horizontal':
            # Horizontal gradient from left to right
            for j in range(dot_size):
                norm_pos = j / (dot_size - 1) if dot_size > 1 else 0
                color_val = int(min_color + (max_color - min_color) * norm_pos)
                gradient[:, j] = np.clip(color_val, min_color, max_color)
        
        elif gradient_type == 'vertical':
            # Vertical gradient from top to bottom
            for i in range(dot_size):
                norm_pos = i / (dot_size - 1) if dot_size > 1 else 0
                color_val = int(min_color + (max_color - min_color) * norm_pos)
                gradient[i, :] = np.clip(color_val, min_color, max_color)
        
        elif gradient_type == 'diagonal':
            # Diagonal gradient from top-left to bottom-right
            for i in range(dot_size):
                for j in range(dot_size):
                    norm_pos = (i + j) / (2 * (dot_size - 1)) if dot_size > 1 else 0
                    color_val = int(min_color + (max_color - min_color) * norm_pos)
                    gradient[i, j] = np.clip(color_val, min_color, max_color)
        
        else:
            raise ValueError(f"Unknown gradient type: {gradient_type}")
        
        return gradient, (min_color, max_color)
    
    def add_gradient_noise(self, image, color_analysis, dot_size=10, num_dots=1, gradient_type='radial'):
        """
        Add gradient noise dots to image
        Returns the image with noise and the noise patterns used
        """
        image_with_noise = image.copy()
        h, w = image.shape[:2]
        
        # Conservative approach: use center 70% region for dot placement
        margin_x, margin_y = int(0.15 * w), int(0.15 * h)
        x_min, y_min = margin_x, margin_y
        x_max, y_max = w - margin_x - dot_size, h - margin_y - dot_size
        
        all_noise_patterns = []
        
        for _ in range(num_dots):
            # Random position within anatomical region
            x = random.randint(x_min, x_max)
            y = random.randint(y_min, y_max)
            
            # Generate gradient noise pattern
            gradient_pattern, color_range = self.generate_gradient_noise_pattern(
                color_analysis, dot_size, gradient_type
            )
            all_noise_patterns.append((gradient_pattern, color_range, gradient_type))
            
            # Add the gradient noise dot
            if len(image.shape) == 3:
                # For colored images, apply to all channels
                for i in range(dot_size):
                    for j in range(dot_size):
                        if y+i < h and x+j < w:
                            color_val = gradient_pattern[i, j]
                            image_with_noise[y+i, x+j] = [color_val, color_val, color_val]
            else:
                # For grayscale images
                if y+dot_size <= h and x+dot_size <= w:
                    image_with_noise[y:y+dot_size, x:x+dot_size] = gradient_pattern
        
        return image_with_noise, all_noise_patterns


In [19]:
class MedicalImageDataset(Dataset):
    def __init__(self, df, root_path, preprocessor, noise_injector, dot_size=10, 
                 gradient_type='radial', transform=None):
        self.df = df.reset_index(drop=True)
        self.root_path = root_path
        self.preprocessor = preprocessor
        self.noise_injector = noise_injector
        self.dot_size = dot_size
        self.gradient_type = gradient_type
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row['ImagePath']
        label = row['noise']
        
        full_path = image_path if os.path.isabs(image_path) else os.path.join(self.root_path, image_path)
        
        processed_image, color_analysis = self.preprocessor.pad_to_square_and_resize(
            full_path, analyze_colors=True
        )
        
        if label == 1:
            processed_image, _ = self.noise_injector.add_gradient_noise(
                processed_image, color_analysis, dot_size=self.dot_size, 
                num_dots=1, gradient_type=self.gradient_type
            )
        
        if len(processed_image.shape) == 3:
            processed_image = cv2.cvtColor(processed_image, cv2.COLOR_BGR2RGB)
        else:
            processed_image = cv2.cvtColor(processed_image, cv2.COLOR_GRAY2RGB)
        
        pil_image = Image.fromarray(processed_image)
        
        if self.transform:
            pil_image = self.transform(pil_image)
        
        return pil_image, label

In [20]:
class ResNet50Classifier:
    def __init__(self, num_classes=2, pretrained=True, input_size=448):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
        
        if pretrained:
            self.model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
            print("Loaded ResNet50 with IMAGENET1K_V2 weights")
        else:
            self.model = models.resnet50(weights=None)
        
        num_features = self.model.fc.in_features
        self.model.fc = nn.Linear(num_features, num_classes)
        
        self.model = self.model.to(self.device)
        
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
        
        print(f"Model configured for {input_size}x{input_size} input")
        print(f"Number of parameters: {sum(p.numel() for p in self.model.parameters()):,}")
    
    def train_model(self, train_loader, val_loader, num_epochs=10, learning_rate=0.001, 
                    checkpoint_dir='checkpoints'):
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=learning_rate)
        
        # Create checkpoint directory
        os.makedirs(checkpoint_dir, exist_ok=True)
        
        best_val_acc = 0.0
        history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
        start_epoch = 0
        
        # Try to load previous checkpoint
        checkpoint_path = os.path.join(checkpoint_dir, 'latest_checkpoint.pth')
        if os.path.exists(checkpoint_path):
            print(f"Found existing checkpoint, loading from {checkpoint_path}")
            checkpoint = torch.load(checkpoint_path)
            self.model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            best_val_acc = checkpoint['best_val_acc']
            history = checkpoint['history']
            print(f"Resuming from epoch {start_epoch}, best val acc: {best_val_acc:.2f}%")
        
        for epoch in range(start_epoch, num_epochs):
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            print("-" * 40)
            
            # Training phase
            self.model.train()
            train_loss = 0.0
            train_correct = 0
            train_total = 0
            
            for images, labels in tqdm(train_loader, desc="Training"):
                images = images.to(self.device)
                labels = labels.to(self.device)
                
                optimizer.zero_grad()
                outputs = self.model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                train_total += labels.size(0)
                train_correct += (predicted == labels).sum().item()
            
            train_loss = train_loss / len(train_loader)
            train_acc = 100 * train_correct / train_total
            
            # Validation phase
            self.model.eval()
            val_loss = 0.0
            val_correct = 0
            val_total = 0
            
            with torch.no_grad():
                for images, labels in tqdm(val_loader, desc="Validation"):
                    images = images.to(self.device)
                    labels = labels.to(self.device)
                    
                    outputs = self.model(images)
                    loss = criterion(outputs, labels)
                    
                    val_loss += loss.item()
                    _, predicted = torch.max(outputs.data, 1)
                    val_total += labels.size(0)
                    val_correct += (predicted == labels).sum().item()
            
            val_loss = val_loss / len(val_loader)
            val_acc = 100 * val_correct / val_total
            
            history['train_loss'].append(train_loss)
            history['train_acc'].append(train_acc)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)
            
            print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
            print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
            
            # Save checkpoint after every epoch
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': self.model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_acc': best_val_acc,
                'history': history,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'train_acc': train_acc,
                'val_acc': val_acc
            }
            checkpoint_path = os.path.join(checkpoint_dir, 'latest_checkpoint.pth')
            torch.save(checkpoint, checkpoint_path)
            print(f"Checkpoint saved to {checkpoint_path}")
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model_path = os.path.join(checkpoint_dir, 'best_resnet50_model.pth')
                torch.save(self.model.state_dict(), best_model_path)
                print(f"✓ Saved best model with validation accuracy: {best_val_acc:.2f}%")
                
                # Also save a backup with epoch number
                epoch_model_path = os.path.join(checkpoint_dir, f'best_model_epoch{epoch+1}.pth')
                torch.save(self.model.state_dict(), epoch_model_path)
        
        return history
    
    def evaluate(self, test_loader):
        """Evaluate the model on test set"""
        self.model.eval()
        all_labels = []
        all_predictions = []
        all_probabilities = []
        
        with torch.no_grad():
            for images, labels in tqdm(test_loader, desc="Testing"):
                images = images.to(self.device)
                labels = labels.to(self.device)
                
                outputs = self.model(images)
                probabilities = torch.softmax(outputs, dim=1)
                _, predicted = torch.max(outputs.data, 1)
                
                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predicted.cpu().numpy())
                all_probabilities.extend(probabilities[:, 1].cpu().numpy())
        
        return np.array(all_labels), np.array(all_predictions), np.array(all_probabilities)

In [21]:
class ImagePipeline:  
    def __init__(self, root_path):
        self.root_path = root_path
        self.preprocessor = ImagePreprocessor()
        self.noise_injector = GradientNoiseInjector()
        self.classifier = None
    
    def load_datasets(self, train_csv_path, test_csv_path):
        print("Loading datasets...")
        self.train_df = pd.read_csv(train_csv_path)
        self.test_df = pd.read_csv(test_csv_path)
        
        print(f"Train dataset shape: {self.train_df.shape}")
        print(f"Test dataset shape: {self.test_df.shape}")
        
        required_cols = ['ImagePath', 'noise']
        for df_name, df in [('train', self.train_df), ('test', self.test_df)]:
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                raise ValueError(f"Missing columns in {df_name} dataset: {missing_cols}")
        
        return self.train_df, self.test_df
    
    def plot_roc_curve(self, labels, probabilities, dot_size, gradient_type, save_path='roc_curve.png'):
        """Plot and save ROC curve"""
        fpr, tpr, thresholds = roc_curve(labels, probabilities)
        roc_auc = sklearn_auc(fpr, tpr)
        
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='darkorange', lw=2, 
                label=f'ROC curve (AUC = {roc_auc:.4f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve - Gradient Noise {dot_size}x{dot_size} ({gradient_type})')
        plt.legend(loc="lower right")
        plt.grid(True, alpha=0.3)
        
        # Save figure
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"ROC curve saved to: {save_path}")
        plt.close()
        
        return roc_auc, fpr, tpr
    
    def run_training(self, train_csv_path, test_csv_path, dot_size=10, gradient_type='radial',
                    batch_size=32, num_epochs=10, learning_rate=0.001, checkpoint_dir='checkpoints'):
       
        print(f"RESNET50 GRADIENT NOISE TRAINING PIPELINE")
        print(f"Image Size: 448x448 | Dot Size: {dot_size}x{dot_size} | Gradient Type: {gradient_type}")
        print(f"Gradient Range: Adaptive from histogram (clipped to 20-235)")
        print(f"Checkpoints will be saved to: {checkpoint_dir}")
        
        train_df, test_df = self.load_datasets(train_csv_path, test_csv_path)
        
        self.classifier = ResNet50Classifier(num_classes=2, pretrained=True, input_size=448)
        
        train_dataset = MedicalImageDataset(
            train_df, self.root_path, self.preprocessor, self.noise_injector,
            dot_size=dot_size, gradient_type=gradient_type, transform=self.classifier.transform
        )
        
        test_dataset = MedicalImageDataset(
            test_df, self.root_path, self.preprocessor, self.noise_injector,
            dot_size=dot_size, gradient_type=gradient_type, transform=self.classifier.transform
        )
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                                 shuffle=True, num_workers=4, pin_memory=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, 
                                shuffle=False, num_workers=4, pin_memory=True)
        
        history = self.classifier.train_model(
            train_loader, test_loader, 
            num_epochs=num_epochs, 
            learning_rate=learning_rate,
            checkpoint_dir=checkpoint_dir
        )
        
        print("\nFINAL EVALUATION ON TEST SET")
        print("=" * 60)
        
        labels, predictions, probabilities = self.classifier.evaluate(test_loader)
        
        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
        
        accuracy = accuracy_score(labels, predictions)
        precision = precision_score(labels, predictions)
        recall = recall_score(labels, predictions)
        f1 = f1_score(labels, predictions)
        auc_score = roc_auc_score(labels, probabilities)
        
        print(f"\nTest Accuracy:  {accuracy:.4f}")
        print(f"Test Precision: {precision:.4f}")
        print(f"Test Recall:    {recall:.4f}")
        print(f"Test F1 Score:  {f1:.4f}")
        print(f"Test AUC:       {auc_score:.4f}")
        
        # Plot ROC curve
        roc_filename = f'roc_curve_gradient{dot_size}_{gradient_type}.png'
        roc_auc, fpr, tpr = self.plot_roc_curve(labels, probabilities, dot_size, 
                                                 gradient_type, save_path=roc_filename)
        
        # Save final results to JSON
        results = {
            'dot_size': dot_size,
            'gradient_type': gradient_type,
            'gradient_range': 'adaptive_20_235',
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'auc': auc_score,
            'roc_auc': roc_auc,
            'fpr': fpr.tolist(),
            'tpr': tpr.tolist(),
            'history': history
        }
        
        results_path = os.path.join(checkpoint_dir, f'final_results_gradient{dot_size}_{gradient_type}.json')
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=4)
        print(f"\nFinal results saved to: {results_path}")
        
        return results

In [22]:
if __name__ == "__main__":
    root_path =  "/home/rmuthy2/central_data/datasets/internal/EmoryCXRv2/ORIGINAL_PNG"
    pipeline = ImagePipeline(root_path)
    
    print("Training with 4x4 pixel noise dots...")
    results_4x4 = pipeline.run_training(
        train_csv_path="/home/rmuthy2/ResNet50/train.csv",
        test_csv_path="/home/rmuthy2/ResNet50/test.csv",
        dot_size=4,  
        gradient_type='radial',  
        batch_size=32,
        num_epochs=10,
        learning_rate=0.001,
        checkpoint_dir='checkpoints_gradient_radial_4'
)

Training with 4x4 pixel noise dots...
RESNET50 GRADIENT NOISE TRAINING PIPELINE
Image Size: 448x448 | Dot Size: 4x4 | Gradient Type: radial
Gradient Range: Adaptive from histogram (clipped to 20-235)
Checkpoints will be saved to: checkpoints_gradient_radial_4
Loading datasets...
Train dataset shape: (109824, 6)
Test dataset shape: (27456, 6)
Using device: cuda
Loaded ResNet50 with IMAGENET1K_V2 weights
Model configured for 448x448 input
Number of parameters: 23,512,130

Epoch 1/10
----------------------------------------


Validation: 100%|██████████| 858/858 [20:24<00:00,  1.43s/it]


Train Loss: 0.1145 | Train Acc: 96.36%
Val Loss: 0.0749 | Val Acc: 97.80%
Checkpoint saved to checkpoints_gradient_radial_4/latest_checkpoint.pth
✓ Saved best model with validation accuracy: 97.80%

Epoch 2/10
----------------------------------------


Validation: 100%|██████████| 858/858 [19:11<00:00,  1.34s/it]


Train Loss: 0.0797 | Train Acc: 97.66%
Val Loss: 0.0734 | Val Acc: 97.84%
Checkpoint saved to checkpoints_gradient_radial_4/latest_checkpoint.pth
✓ Saved best model with validation accuracy: 97.84%

Epoch 3/10
----------------------------------------


...

Train Loss: 0.0509 | Train Acc: 98.58%
Val Loss: 0.0551 | Val Acc: 98.47%
Checkpoint saved to checkpoints_gradient_radial_4/latest_checkpoint.pth

Epoch 10/10
----------------------------------------


Validation: 100%|██████████| 858/858 [19:13<00:00,  1.34s/it]


Train Loss: 0.0494 | Train Acc: 98.63%
Val Loss: 0.0589 | Val Acc: 98.33%
Checkpoint saved to checkpoints_gradient_radial_4/latest_checkpoint.pth

FINAL EVALUATION ON TEST SET


Testing: 100%|██████████| 858/858 [19:44<00:00,  1.38s/it]



Test Accuracy:  0.9835
Test Precision: 0.9969
Test Recall:    0.9701
Test F1 Score:  0.9833
Test AUC:       0.9953
ROC curve saved to: roc_curve_gradient4_radial.png

Final results saved to: checkpoints_gradient_radial_4/final_results_gradient4_radial.json
